# Visualization Theme and Export

AbaQuant's visualization layer supports a **global theme** you configure
once, plus a **temporary theme context manager** for one-off overrides
(e.g., switching to Plotly and HTML export just for a single figure).

**Sections:**
1. Configure a reusable global theme
2. Create a themed figure, then temporarily override the theme
3. Reset back to the default theme


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
from pathlib import Path

from abaquant.derivatives.models import BlackScholesMertonModel
from abaquant.visualization import (
    VisualizationError,
    VisualizationTheme,
    configure_visualization,
    get_visualization_theme,
    reset_visualization_theme,
    visualization_theme,
)

output_directory = Path("generated_figures/visualization_theme")
output_directory.mkdir(parents=True, exist_ok=True)

## 1. Configure a reusable global theme

`configure_visualization()` sets the theme that every subsequent
`.visualize()` call will use by default, until reset.


In [ ]:
theme = configure_visualization(
    VisualizationTheme(
        backend="matplotlib",
        color_sequence=("#0F4C81", "#E07A5F", "#3D9970"),
        background_color="#FAFAFA",
        paper_color="#FAFAFA",
        grid_color="#CBD5E1",
        font_family="DejaVu Sans",
        figure_size=(10.0, 5.8),
        dpi=140,
        line_width=2.5,
        marker_size=6.0,
        save_directory=output_directory,
        save_format="png",
        auto_save=False,
        filename_prefix="theme_example",
    )
)
print(f"backend={theme.backend}, font_family={theme.font_family}, dpi={theme.dpi}")

## 2. Themed figures and a temporary override

The global figure below uses the theme configured above. The
`visualization_theme(...)` context manager then applies a *different*
theme (Plotly + HTML export) just inside the `with` block, without
disturbing the global configuration.


In [ ]:
try:
    model = BlackScholesMertonModel(100.0, 100.0, 1.0, 0.05, 0.20)
    global_figure = model.visualize(chart="price_profile", filename="global_theme_profile")

    with visualization_theme(
        backend="plotly",
        color_sequence=("#5B2C6F", "#1F618D"),
        save_format="html",
        save_directory=output_directory,
    ):
        temporary_figure = model.visualize(chart="payoff", filename="temporary_plotly_payoff")

    active_theme = get_visualization_theme()
    print(f"Global figure type:    {type(global_figure).__name__}")
    print(f"Temporary figure type: {type(temporary_figure).__name__}")
    print(f"Active backend after context exits: {active_theme.backend}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## 3. Reset to the built-in default theme

Always reset at the end of a notebook/script so later cells don't inherit a custom theme unintentionally.


In [ ]:
reset_visualization_theme()
print("Theme reset to AbaQuant defaults.")

## Takeaway

Use the global theme for consistent, notebook-wide styling, and the
`visualization_theme()` context manager for one-off backend or export
overrides. `auto_save=True` on the theme will make every `.visualize()`
call write a file automatically; the default (`False`) keeps figures purely
in-memory unless you pass `filename=` explicitly.
